# Gate 2-A/B. 모집요강 반영규칙 후보 수집

이 노트북은 Gate 1 크롤 산출물과 HTML 캐시만 사용해 Gate 2의 시작점을 만든다.

범위:

- 기존 `raw_html/`에서 모집요강성 규칙 후보 표 추출
- 수능 반영비율, 환산점수 산출방법, 영어/한국사 등급 변환, 가산점, 전형요소 관련 text snippet 추출
- 대학별 Gate 2 수집·수동검토 우선순위 산정
- 입학처/모집요강 외부 재수집이 필요한 대학 분리

범위 밖:

- 웹 재크롤링
- PDF 다운로드
- 모집요강 원문 확정 파싱
- 학과 crosswalk 확정
- H1/H2 모델링

In [1]:
from pathlib import Path
import re
import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience")
BASE_DIR = PROJECT_ROOT / "workbook/p2/p2_2/data/crawl_2024_admission"
EDA_DIR = BASE_DIR / "eda_outputs"
RAW_HTML_DIR = BASE_DIR / "raw_html"
OUTPUT_DIR = BASE_DIR / "gate2_outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SEED_PATH = BASE_DIR / "00_crawl_seed_university_2024.csv"
REGISTRY_PATH = BASE_DIR / "01_crawl_source_registry.csv"
EDA_READY_PATH = EDA_DIR / "03_admission_result_eda_ready_2024.parquet"
QUALITY_DASHBOARD_PATH = EDA_DIR / "eda_16_university_quality_dashboard.csv"
SCORE_COVERAGE_PATH = EDA_DIR / "eda_13_university_score_coverage.csv"
CROSSWALK_PATH = EDA_DIR / "eda_21_crosswalk_difficulty.csv"

RUN_STARTED_AT = datetime.now()
AUDIT_LOG = []

plt.rcParams.update(
    {
        "font.family": "NanumGothic",
        "axes.unicode_minus": False,
        "figure.dpi": 120,
        "savefig.dpi": 170,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")

def audit(step, dataset, action, rows_before=None, rows_after=None, status="INFO", note=""):
    AUDIT_LOG.append(
        {
            "step": step,
            "dataset": dataset,
            "action": action,
            "rows_before": rows_before,
            "rows_after": rows_after,
            "status": status,
            "note": note,
            "executed_at": now_iso(),
        }
    )

def save_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    audit("export", path.name, "save_csv", rows_before=len(df), rows_after=len(df), status="PASS", note=str(path))
    return path

def save_fig(fig, name: str):
    path = FIGURE_DIR / name
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    audit("export", name, "save_png", status="PASS", note=str(path))
    return path

def normalize_text(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()

def resolve_raw_file_path(value) -> Path:
    text = normalize_text(value)
    p = Path(text)
    if p.is_absolute():
        return p
    for candidate in [PROJECT_ROOT / text, BASE_DIR / p.name, RAW_HTML_DIR / p.name]:
        if candidate.exists():
            return candidate
    return PROJECT_ROOT / text

display(
    pd.DataFrame(
        [
            {"key": "BASE_DIR", "value": str(BASE_DIR)},
            {"key": "OUTPUT_DIR", "value": str(OUTPUT_DIR)},
            {"key": "RUN_STARTED_AT", "value": RUN_STARTED_AT.isoformat(timespec="seconds")},
        ]
    )
)
audit("setup", "environment", "initialize", status="PASS")

,key,value
0,BASE_DIR,/home/sieg/projects-wsl/SBS_dataScience/workbo...
1,OUTPUT_DIR,/home/sieg/projects-wsl/SBS_dataScience/workbo...
2,RUN_STARTED_AT,2026-07-10T13:55:49


In [2]:
seed = pd.read_csv(SEED_PATH)
registry = pd.read_csv(REGISTRY_PATH)
admission_eda = pd.read_parquet(EDA_READY_PATH)
quality = pd.read_csv(QUALITY_DASHBOARD_PATH)
score_coverage = pd.read_csv(SCORE_COVERAGE_PATH)
crosswalk = pd.read_csv(CROSSWALK_PATH)

seed["adiga_univ_code_str"] = seed["adiga_univ_code"].map(lambda x: str(int(x)).zfill(7) if pd.notna(x) else "")
seed_lookup = (
    seed.sort_values(["target_institution_flag", "crawl_priority"], ascending=[False, True])
    .drop_duplicates("univ_id", keep="first")
    [["univ_id", "univ_name_std", "adiga_univ_code_str", "branch_status", "seed_status" if "seed_status" in seed.columns else "target_institution_flag"]]
    .copy()
)
if "seed_status" not in seed_lookup.columns:
    seed_lookup["seed_status"] = np.where(seed_lookup["target_institution_flag"], "target_or_primary", "excluded_or_secondary")
    seed_lookup = seed_lookup.drop(columns=["target_institution_flag"])

registry = registry.merge(seed_lookup[["univ_id", "univ_name_std", "adiga_univ_code_str"]], on="univ_id", how="left")
registry["raw_html_abs_path"] = registry["raw_file_path"].map(lambda x: str(resolve_raw_file_path(x)))
registry["raw_html_exists"] = registry["raw_html_abs_path"].map(lambda x: Path(x).exists())

display(
    pd.DataFrame(
        [
            {"dataset": "seed", "shape": str(seed.shape)},
            {"dataset": "registry", "shape": str(registry.shape)},
            {"dataset": "admission_eda", "shape": str(admission_eda.shape)},
            {"dataset": "quality", "shape": str(quality.shape)},
            {"dataset": "score_coverage", "shape": str(score_coverage.shape)},
            {"dataset": "crosswalk", "shape": str(crosswalk.shape)},
        ]
    )
)
audit("load", "inputs", "load_gate2_inputs", rows_after=len(registry), status="PASS")

,dataset,shape
0,seed,"(52, 13)"
1,registry,"(51, 22)"
2,admission_eda,"(3096, 64)"
3,quality,"(51, 27)"
4,score_coverage,"(51, 14)"
5,crosswalk,"(51, 12)"


In [3]:
KEYWORD_GROUPS = {
    "csat_reflection_ratio": [
        "수능 반영", "수능반영", "반영비율", "반영 비율", "영역별 반영", "국어", "수학", "영어", "탐구",
    ],
    "score_conversion": [
        "환산점수", "산출방법", "산출 방법", "점수산출", "점수 산출", "백분위", "표준점수", "변환표준점수",
    ],
    "english_korean_history": [
        "영어 등급", "한국사", "등급별", "감점", "가산점", "등급 환산", "등급별 점수",
    ],
    "selection_element": [
        "전형요소", "일괄합산", "수능 100", "수능100", "실기", "학생부", "면접",
    ],
    "minimum_standard": [
        "수능최저", "최저학력", "최저 학력",
    ],
    "admission_guide_link": [
        "모집요강", "정시모집", "정시 모집", "전형계획", "입학처", "다운로드", "pdf",
    ],
}
SUBJECT_PATTERNS = {
    "has_korean": r"국어",
    "has_math": r"수학|미적분|기하|확률",
    "has_english": r"영어",
    "has_inquiry": r"탐구|사탐|과탐|사회탐구|과학탐구",
    "has_history": r"한국사",
    "has_second_language": r"제2외국어|한문",
}
PERCENT_RE = re.compile(r"(?<!\d)(\d{1,3}(?:\.\d+)?)\s*%")
NUMBER_RE = re.compile(r"(?<!\d)(\d{1,4}(?:\.\d+)?)(?!\d)")

def keyword_hits(text: str) -> dict:
    return {
        group: sum(1 for kw in keywords if kw.lower() in text.lower())
        for group, keywords in KEYWORD_GROUPS.items()
    }

def candidate_type_from_hits(hits: dict) -> str:
    ordered = [
        "csat_reflection_ratio",
        "score_conversion",
        "english_korean_history",
        "selection_element",
        "minimum_standard",
        "admission_guide_link",
    ]
    positives = [(g, hits.get(g, 0)) for g in ordered if hits.get(g, 0) > 0]
    return max(positives, key=lambda x: x[1])[0] if positives else "other"

def subject_flags(text: str) -> dict:
    return {name: bool(re.search(pattern, text)) for name, pattern in SUBJECT_PATTERNS.items()}

def compact_excerpt(text: str, limit: int = 900) -> str:
    text = normalize_text(text)
    return text[:limit] + ("..." if len(text) > limit else "")

def table_grid(table):
    rows = []
    for tr in table.find_all("tr"):
        cells = [normalize_text(cell.get_text(" ", strip=True)) for cell in tr.find_all(["th", "td"])]
        if any(cells):
            rows.append(cells)
    return rows

def table_score(text: str, hits: dict, row_n: int, col_n: int) -> int:
    score = 0
    score += hits.get("csat_reflection_ratio", 0) * 3
    score += hits.get("score_conversion", 0) * 2
    score += hits.get("english_korean_history", 0) * 2
    score += hits.get("selection_element", 0) * 2
    score += hits.get("minimum_standard", 0)
    if PERCENT_RE.search(text):
        score += 3
    if row_n >= 2 and col_n >= 2:
        score += 1
    if "경쟁률" in text and "충원" in text and "모집단위" in text:
        score -= 5
    return int(score)

def snippet_windows(text: str, group: str, keywords: list[str], window: int = 220, max_per_group: int = 4):
    lowered = text.lower()
    windows = []
    seen = set()
    for keyword in keywords:
        pos = lowered.find(keyword.lower())
        while pos >= 0 and len(windows) < max_per_group:
            start = max(0, pos - window)
            end = min(len(text), pos + len(keyword) + window)
            snippet = compact_excerpt(text[start:end], limit=520)
            key = sha256_text(snippet)[:16]
            if key not in seen:
                windows.append((keyword, snippet, start, end))
                seen.add(key)
            pos = lowered.find(keyword.lower(), pos + len(keyword))
    return windows

In [4]:
table_rows = []
snippet_rows = []
link_rows = []

for _, reg in registry.iterrows():
    html_path = Path(reg["raw_html_abs_path"])
    univ_id = reg["univ_id"]
    univ_name = reg["univ_name_std"]
    if not html_path.exists():
        audit("parse_html", univ_id, "missing_html", status="FAIL", note=str(html_path))
        continue
    html = html_path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")
    full_text = normalize_text(soup.get_text(" ", strip=True))

    for a_idx, a in enumerate(soup.find_all("a")):
        text = normalize_text(a.get_text(" ", strip=True))
        href = normalize_text(a.get("href", ""))
        combo = f"{text} {href}"
        hits = keyword_hits(combo)
        if hits["admission_guide_link"] > 0 or re.search(r"pdf|download|file|attach", combo, re.I):
            link_rows.append(
                {
                    "univ_id": univ_id,
                    "univ_name_std": univ_name,
                    "source_id": reg["source_id"],
                    "anchor_index": a_idx,
                    "anchor_text": text,
                    "href": href,
                    "candidate_type": candidate_type_from_hits(hits),
                    "keyword_hit_json": json.dumps(hits, ensure_ascii=False),
                }
            )

    for t_idx, table in enumerate(soup.find_all("table")):
        rows = table_grid(table)
        if not rows:
            continue
        row_n = len(rows)
        col_n = max((len(r) for r in rows), default=0)
        table_text = normalize_text(" | ".join(" / ".join(r) for r in rows))
        hits = keyword_hits(table_text)
        score = table_score(table_text, hits, row_n, col_n)
        result_table_like = "경쟁률" in table_text and "충원" in table_text and "모집단위" in table_text
        has_pct = bool(PERCENT_RE.search(table_text))
        flags = subject_flags(table_text)
        if score >= 4 or (has_pct and sum(flags.values()) >= 2) or hits["score_conversion"] >= 2:
            table_rows.append(
                {
                    "univ_id": univ_id,
                    "univ_name_std": univ_name,
                    "source_id": reg["source_id"],
                    "html_file": html_path.name,
                    "table_index": t_idx,
                    "row_n": row_n,
                    "max_col_n": col_n,
                    "candidate_score": score,
                    "candidate_type": candidate_type_from_hits(hits),
                    "review_priority": "high"
                    if score >= 10 and not result_table_like
                    else ("medium" if score >= 6 else "low"),
                    "result_table_like": result_table_like,
                    "has_percent_token": has_pct,
                    "percent_tokens": "; ".join(PERCENT_RE.findall(table_text)[:20]),
                    "number_tokens": "; ".join(NUMBER_RE.findall(table_text)[:30]),
                    **flags,
                    "keyword_hit_json": json.dumps(hits, ensure_ascii=False),
                    "table_text_sha256": sha256_text(table_text),
                    "table_text_excerpt": compact_excerpt(table_text, 1200),
                }
            )

    for group, keywords in KEYWORD_GROUPS.items():
        if group == "admission_guide_link":
            continue
        for keyword, snippet, start, end in snippet_windows(full_text, group, keywords):
            hits = keyword_hits(snippet)
            flags = subject_flags(snippet)
            snippet_rows.append(
                {
                    "univ_id": univ_id,
                    "univ_name_std": univ_name,
                    "source_id": reg["source_id"],
                    "html_file": html_path.name,
                    "candidate_type": group,
                    "matched_keyword": keyword,
                    "char_start": start,
                    "char_end": end,
                    "has_percent_token": bool(PERCENT_RE.search(snippet)),
                    "percent_tokens": "; ".join(PERCENT_RE.findall(snippet)[:20]),
                    **flags,
                    "keyword_hit_json": json.dumps(hits, ensure_ascii=False),
                    "snippet_sha256": sha256_text(snippet),
                    "snippet": snippet,
                }
            )

guide_links = pd.DataFrame(link_rows)
rule_table_candidates = pd.DataFrame(table_rows)
rule_text_snippets = pd.DataFrame(snippet_rows)

save_csv(guide_links, OUTPUT_DIR / "gate2_00_admission_guide_links.csv")
save_csv(rule_table_candidates, OUTPUT_DIR / "gate2_01_rule_table_candidates.csv")
save_csv(rule_text_snippets, OUTPUT_DIR / "gate2_02_rule_text_snippets.csv")
audit("parse_html", "raw_html", "extract_rule_candidates", rows_after=len(rule_table_candidates), status="PASS")

display(pd.DataFrame([
    {"artifact": "guide_links", "rows": len(guide_links)},
    {"artifact": "rule_table_candidates", "rows": len(rule_table_candidates)},
    {"artifact": "rule_text_snippets", "rows": len(rule_text_snippets)},
]))
display(rule_table_candidates.head(10))

,artifact,rows
0,guide_links,102
1,rule_table_candidates,1306
2,rule_text_snippets,1008


,univ_id,univ_name_std,source_id,html_file,table_index,row_n,max_col_n,candidate_score,candidate_type,review_priority,...,number_tokens,has_korean,has_math,has_english,has_inquiry,has_history,has_second_language,keyword_hit_json,table_text_sha256,table_text_excerpt
0,U0000019,서울대학교,SRC_0000019_2025_0001,0000019_2025.html,0,11,4,22,csat_reflection_ratio,high,...,2025; 2024; 1; 2.5; 50; 1; 50; 4; 2; 6; 1; 2.5...,True,True,True,True,False,False,"{""csat_reflection_ratio"": 4, ""score_conversion...",d0a8ca7886434883f502fc9ebc57e0a52cfc9973d0fab5...,구분 / 2025 학년도 / 2024 학년도 / 비고 | 공통 / [ 신설 ] 농업...
1,U0000019,서울대학교,SRC_0000019_2025_0001,0000019_2025.html,1,5,2,8,selection_element,medium,...,2; 1; 3; 2; 1; 1; 2; 2; 1; 5; 1; 2; 2; 1,False,False,False,False,False,False,"{""csat_reflection_ratio"": 0, ""score_conversion...",b132ede524c451a03dfed40b04b1f56ec425103578b7ae...,"전형별 주요사항 | 수시 지역균형전형 / - 다양한 지역적 , 사회·경제적 배경하에..."
2,U0000019,서울대학교,SRC_0000019_2025_0001,0000019_2025.html,2,10,7,8,selection_element,medium,...,506; 1; 3; 100; 2; 70; 1; 30; 1; 499; 1; 2; 10...,False,False,False,False,False,False,"{""csat_reflection_ratio"": 1, ""score_conversion...",9038af25e55a631644ca953aa5c235f865ac53a9b61578...,구분 / 전형개요 / 비고 | 모집 인원 / 선발방법 / 전형요소 및 반영비율 | ...
3,U0000019,서울대학교,SRC_0000019_2025_0001,0000019_2025.html,3,4,2,4,csat_reflection_ratio,low,...,,False,False,False,True,False,False,"{""csat_reflection_ratio"": 1, ""score_conversion...",3f273afa37d92498d32674a89e0b4aac7f0607a1bc63e7...,평가요소 / 평가 내용 | 학업역량 / · 평가 요소 의미 - 폭넓은 지식을 깊이 ...
4,U0000019,서울대학교,SRC_0000019_2025_0001,0000019_2025.html,4,7,15,4,csat_reflection_ratio,low,...,7; 3; 3; 7; 7; 3; 3; 7; 7; 3; 3; 7,False,False,False,False,False,False,"{""csat_reflection_ratio"": 1, ""score_conversion...",edaf2853b042e08dd5b15422f408fec77ede92d20e4d1b...,구분 / 내용 | 서류평가 평가요소별 배점 / 구분 학업역량 학업태도 학업외소양 최...
5,U0000019,서울대학교,SRC_0000019_2025_0001,0000019_2025.html,7,12,15,9,csat_reflection_ratio,medium,...,2; 15; 30; 45; 100; 2024,False,True,False,True,False,False,"{""csat_reflection_ratio"": 2, ""score_conversion...",29be4117cb025b89798fdef43a570ba474d2e6f95bf1ca...,해당전형 / 수시 일반전형 | 진행방법 / 평가위원 2 인 이상 / 개별면접 | 면...
6,U0000019,서울대학교,SRC_0000019_2025_0001,0000019_2025.html,8,6,2,7,csat_reflection_ratio,medium,...,,False,True,False,True,False,False,"{""csat_reflection_ratio"": 2, ""score_conversion...",e57c8d970fd063f5ab51532a4217617aa66431c6e77902...,"수학 ( 인문 ) / 수학 , 수학Ⅰ , 수학Ⅱ , 확률과 통계 | 수학 ( 자연 ..."
7,U0000019,서울대학교,SRC_0000019_2025_0001,0000019_2025.html,9,11,37,4,csat_reflection_ratio,low,...,1; 2; 3; 1; 2; 1; 2; 1; 2; 3; 300; 1; 20; 3; 1...,False,False,False,False,False,False,"{""csat_reflection_ratio"": 1, ""score_conversion...",2c8a588f2a396e8c5629380a18f530ccc00563dc2f8a3c...,전 형 / 교과성적 / 1 학년 / 2 학년 / 3 학년 / 비고 | 1 학기 / ...
8,U0000019,서울대학교,SRC_0000019_2025_0001,0000019_2025.html,12,61,7,10,csat_reflection_ratio,medium,...,50; 70; 27; 4.81; 1; 3; 1.20; 1.31; 9; 2.33; 1...,True,True,True,False,False,False,"{""csat_reflection_ratio"": 3, ""score_conversion...",53210eec90ffa5dcea83430f5b1d5e83dac386e9ba8b1f...,모집단위 / 수시 지역균형전형 | 모집 인원 / 경쟁률 / 충원 합격 순위 / 최종...
9,U0000019,서울대학교,SRC_0000019_2025_0001,0000019_2025.html,13,82,7,10,csat_reflection_ratio,medium,...,50; 70; 9; 12.11; 1; 0; 1.75; 2.35; 9; 8.78; 1...,True,True,True,False,False,False,"{""csat_reflection_ratio"": 3, ""score_conversion...",d019143bd2fe86c689092f4cc6ab2e909ff706c1d7670d...,모집단위 / 수시 일반전형 | 모집 인원 / 경쟁률 / 충원 합격 순위 / 최종등록...


In [5]:
if len(rule_table_candidates):
    type_summary = (
        rule_table_candidates.groupby("candidate_type", dropna=False)
        .agg(
            table_candidate_n=("table_text_sha256", "count"),
            university_n=("univ_id", "nunique"),
            high_priority_n=("review_priority", lambda s: int((s == "high").sum())),
            percent_table_n=("has_percent_token", "sum"),
            result_table_like_n=("result_table_like", "sum"),
            median_score=("candidate_score", "median"),
            max_score=("candidate_score", "max"),
        )
        .reset_index()
        .sort_values(["high_priority_n", "table_candidate_n"], ascending=False)
    )
else:
    type_summary = pd.DataFrame()

table_pivot = (
    rule_table_candidates.pivot_table(
        index="univ_id",
        columns="candidate_type",
        values="table_text_sha256",
        aggfunc="count",
        fill_value=0,
    )
    if len(rule_table_candidates)
    else pd.DataFrame(index=registry["univ_id"].unique())
)
for col in ["csat_reflection_ratio", "score_conversion", "english_korean_history", "selection_element", "minimum_standard"]:
    if col not in table_pivot.columns:
        table_pivot[col] = 0
table_pivot = table_pivot.reset_index()

high_priority = (
    rule_table_candidates.loc[rule_table_candidates["review_priority"].eq("high")]
    .groupby("univ_id")
    .size()
    .rename("high_priority_rule_table_n")
    .reset_index()
    if len(rule_table_candidates)
    else pd.DataFrame({"univ_id": registry["univ_id"].unique(), "high_priority_rule_table_n": 0})
)
snippet_pivot = (
    rule_text_snippets.pivot_table(
        index="univ_id",
        columns="candidate_type",
        values="snippet_sha256",
        aggfunc="count",
        fill_value=0,
    )
    if len(rule_text_snippets)
    else pd.DataFrame(index=registry["univ_id"].unique())
)
snippet_pivot = snippet_pivot.add_prefix("snippet_").reset_index()
link_counts = (
    guide_links.groupby("univ_id").size().rename("guide_link_candidate_n").reset_index()
    if len(guide_links)
    else pd.DataFrame({"univ_id": registry["univ_id"].unique(), "guide_link_candidate_n": 0})
)

priority = registry[["univ_id", "univ_name_std", "source_id", "source_url", "raw_html_exists"]].copy()
priority = priority.merge(table_pivot, on="univ_id", how="left")
priority = priority.merge(snippet_pivot, on="univ_id", how="left")
priority = priority.merge(high_priority, on="univ_id", how="left")
priority = priority.merge(link_counts, on="univ_id", how="left")
priority = priority.merge(
    quality[["univ_id", "quality_status", "raw_row_n", "parser_anomaly_row_n", "tier_a_row_n", "tier_b_row_n", "tier_c_row_n", "tier_d_row_n"]],
    on="univ_id",
    how="left",
)
priority = priority.merge(
    score_coverage[["univ_id", "percentile_70cut_available_rate", "univ_score_70cut_available_rate", "univ_score_ratio_available_rate"]],
    on="univ_id",
    how="left",
)
priority = priority.merge(
    crosswalk[["univ_id", "crosswalk_difficulty_score", "crosswalk_priority"]],
    on="univ_id",
    how="left",
)
priority = priority.fillna(0)

priority["has_csat_rule_candidate"] = priority["csat_reflection_ratio"].gt(0) | priority.get("snippet_csat_reflection_ratio", 0).gt(0)
priority["has_score_rule_candidate"] = priority["score_conversion"].gt(0) | priority.get("snippet_score_conversion", 0).gt(0)
priority["has_guide_link_candidate"] = priority["guide_link_candidate_n"].gt(0)
priority["rule_cache_status"] = np.select(
    [
        priority["has_csat_rule_candidate"] & priority["has_score_rule_candidate"],
        priority["has_csat_rule_candidate"] | priority["has_score_rule_candidate"] | priority["has_guide_link_candidate"],
    ],
    ["CACHE_RULE_CANDIDATES", "PARTIAL_CACHE_CANDIDATES"],
    default="NEED_EXTERNAL_GUIDE",
)
priority["manual_priority_score"] = (
    priority["quality_status"].eq("FAIL").astype(int) * 4
    + priority["percentile_70cut_available_rate"].lt(0.5).astype(int) * 3
    + priority["crosswalk_priority"].eq("high").astype(int) * 2
    + (~priority["has_csat_rule_candidate"]).astype(int) * 2
    + (~priority["has_score_rule_candidate"]).astype(int) * 1
    + priority["parser_anomaly_row_n"].gt(0).astype(int)
)
priority["gate2_action"] = np.select(
    [
        priority["rule_cache_status"].eq("NEED_EXTERNAL_GUIDE"),
        priority["manual_priority_score"].ge(6),
        priority["rule_cache_status"].eq("PARTIAL_CACHE_CANDIDATES"),
    ],
    ["입학처/모집요강 외부 수집 우선", "캐시 후보 수동검토 최우선", "캐시 후보 검토 후 부족분 외부수집"],
    default="캐시 후보에서 반영규칙 후보 추출",
)
priority = priority.sort_values(["manual_priority_score", "raw_row_n"], ascending=[False, False])

if len(rule_table_candidates):
    type_rank = {
        "csat_reflection_ratio": 0,
        "score_conversion": 1,
        "english_korean_history": 2,
        "selection_element": 3,
        "minimum_standard": 4,
        "other": 5,
    }
    review_queue = rule_table_candidates.loc[~rule_table_candidates["result_table_like"]].copy()
    review_queue["candidate_type_rank"] = review_queue["candidate_type"].map(type_rank).fillna(9).astype(int)
    review_queue["review_priority_rank"] = review_queue["review_priority"].map({"high": 0, "medium": 1, "low": 2}).fillna(3).astype(int)
    review_queue = review_queue.sort_values(
        ["univ_id", "review_priority_rank", "candidate_type_rank", "candidate_score", "table_index"],
        ascending=[True, True, True, False, True],
    )
    review_queue["candidate_order_in_university"] = review_queue.groupby("univ_id").cumcount() + 1
    review_queue = review_queue.loc[review_queue["candidate_order_in_university"].le(5)].copy()
    review_queue["review_task"] = np.select(
        [
            review_queue["candidate_type"].eq("csat_reflection_ratio"),
            review_queue["candidate_type"].eq("score_conversion"),
            review_queue["candidate_type"].eq("english_korean_history"),
            review_queue["candidate_type"].eq("selection_element"),
        ],
        [
            "수능 영역별 반영비율/지정응시영역 확인",
            "환산점수 산출방법/활용지표 확인",
            "영어·한국사 등급 변환/가산·감점 확인",
            "전형요소 및 수능 100% 여부 확인",
        ],
        default="기타 규칙 후보 검토",
    )
    review_queue = review_queue[
        [
            "univ_id",
            "univ_name_std",
            "candidate_order_in_university",
            "review_task",
            "candidate_type",
            "candidate_score",
            "review_priority",
            "table_index",
            "row_n",
            "max_col_n",
            "has_percent_token",
            "percent_tokens",
            "has_korean",
            "has_math",
            "has_english",
            "has_inquiry",
            "has_history",
            "table_text_excerpt",
        ]
    ].sort_values(["univ_name_std", "candidate_order_in_university"])
else:
    review_queue = pd.DataFrame()

save_csv(type_summary, OUTPUT_DIR / "gate2_04_candidate_type_summary.csv")
save_csv(priority, OUTPUT_DIR / "gate2_03_university_rule_collection_priority.csv")
save_csv(review_queue, OUTPUT_DIR / "gate2_07_top_rule_candidates_for_review.csv")
display(type_summary)
display(priority.head(20))
display(review_queue.head(20))

,candidate_type,table_candidate_n,university_n,high_priority_n,percent_table_n,result_table_like_n,median_score,max_score
0,csat_reflection_ratio,849,51,458,542,249,12.0,52
5,selection_element,339,51,174,183,4,10.0,27
4,score_conversion,46,30,25,12,0,10.0,48
1,english_korean_history,15,12,4,3,0,7.0,27
2,minimum_standard,20,17,2,14,0,8.0,13
3,other,37,19,0,37,0,4.0,4


,univ_id,univ_name_std,source_id,source_url,raw_html_exists,csat_reflection_ratio,english_korean_history,minimum_standard,other,score_conversion,...,univ_score_70cut_available_rate,univ_score_ratio_available_rate,crosswalk_difficulty_score,crosswalk_priority,has_csat_rule_candidate,has_score_rule_candidate,has_guide_link_candidate,rule_cache_status,manual_priority_score,gate2_action
1,U0000149,연세대학교,SRC_0000149_2025_0002,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,True,27,1,0,0,1,...,0.344681,0.000000,0.051064,low,True,True,True,CACHE_RULE_CANDIDATES,7,캐시 후보 수동검토 최우선
35,U0000102,동덕여자대학교,SRC_0000102_2025_0036,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,True,18,0,0,0,1,...,0.000000,0.000000,0.226744,medium,True,True,True,CACHE_RULE_CANDIDATES,7,캐시 후보 수동검토 최우선
25,U0000074,광운대학교,SRC_0000074_2025_0026,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,True,18,0,0,1,0,...,0.000000,0.000000,0.184615,low,True,True,True,CACHE_RULE_CANDIDATES,7,캐시 후보 수동검토 최우선
15,U0000141,숙명여자대학교,SRC_0000141_2025_0016,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,True,11,0,1,1,0,...,0.000000,0.000000,0.071429,low,True,True,True,CACHE_RULE_CANDIDATES,7,캐시 후보 수동검토 최우선
8,U0000040,서울시립대학교,SRC_0000040_2025_0009,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,True,15,0,1,0,1,...,1.000000,0.000000,0.137500,low,True,True,True,CACHE_RULE_CANDIDATES,7,캐시 후보 수동검토 최우선
43,U0000013,부경대학교,SRC_0000013_2025_0044,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,True,20,0,0,4,0,...,0.885246,0.885246,0.114754,low,True,True,True,CACHE_RULE_CANDIDATES,5,캐시 후보에서 반영규칙 후보 추출
29,U0000046,가톨릭대학교,SRC_0000046_2025_0030,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,True,14,0,1,1,1,...,1.000000,1.000000,1.842105,high,True,True,True,CACHE_RULE_CANDIDATES,5,캐시 후보에서 반영규칙 후보 추출
20,U0000036,서울과학기술대학교,SRC_0000036_2025_0021,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,True,13,0,0,0,1,...,1.000000,1.000000,0.841463,high,True,True,True,CACHE_RULE_CANDIDATES,2,캐시 후보에서 반영규칙 후보 추출
18,U0000138,세종대학교,SRC_0000138_2025_0019,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,True,21,0,0,0,1,...,1.000000,1.000000,0.967742,high,True,True,True,CACHE_RULE_CANDIDATES,2,캐시 후보에서 반영규칙 후보 추출
34,U0000099,덕성여자대학교,SRC_0000099_2025_0035,https://www.adiga.kr/ucp/uvt/uni/univDetailSel...,True,25,0,1,0,0,...,1.000000,1.000000,0.857143,high,True,True,True,CACHE_RULE_CANDIDATES,2,캐시 후보에서 반영규칙 후보 추출


,univ_id,univ_name_std,candidate_order_in_university,review_task,candidate_type,candidate_score,review_priority,table_index,row_n,max_col_n,has_percent_token,percent_tokens,has_korean,has_math,has_english,has_inquiry,has_history,table_text_excerpt
889,U0000063,가천대학교,1,수능 영역별 반영비율/지정응시영역 확인,csat_reflection_ratio,32,high,36,13,69,True,100; 35; 25; 35; 25; 60; 40; 35; 25; 35; 25; 6...,True,True,True,True,True,수능 성적 산출방법 | 일반전형 ( 수능 100%) 계열 / 모집단위 수능영역별 반...
878,U0000063,가천대학교,2,수능 영역별 반영비율/지정응시영역 확인,csat_reflection_ratio,25,high,23,26,97,True,100; 100; 100; 40; 30; 20; 10; 25; 50; 100; 10...,True,True,True,False,False,학생부 교과성적 산출방법 | 1. 반영 비율 학년별 반영비율 요소별 반영비율 1 학...
865,U0000063,가천대학교,3,수능 영역별 반영비율/지정응시영역 확인,csat_reflection_ratio,24,high,0,9,4,True,100; 80; 20; 100; 50; 50; 100; 50; 50; 100; 40...,True,True,True,True,False,구분 / 2025 학년도 / 2024 학년도 / 비고 | 전형방법 / 논술전형 논술...
890,U0000063,가천대학교,4,수능 영역별 반영비율/지정응시영역 확인,csat_reflection_ratio,24,high,37,7,7,True,35; 25; 35; 25,True,True,True,True,True,계열 / 모집단위 / 수능영역별 반영비율 (%) / 한국사 | 국어 / 수학 / 영...
875,U0000063,가천대학교,5,수능 영역별 반영비율/지정응시영역 확인,csat_reflection_ratio,20,high,20,7,3,True,40; 30; 20; 10; 25; 50; 40; 30; 20; 10; 25; 50,True,True,True,False,False,"구분 / 공통과목 , 일반선택과목 / 진로선택과목 | 반영교과 / - 인문계열 : ..."
784,U0000046,가톨릭대학교,1,수능 영역별 반영비율/지정응시영역 확인,csat_reflection_ratio,49,high,35,19,113,False,,True,True,True,True,True,수능 성적 산출방법 | 1. 대학수학능력시험 성적반영 방법 가 . 기본사항 1) 대...
763,U0000046,가톨릭대학교,2,수능 영역별 반영비율/지정응시영역 확인,csat_reflection_ratio,31,high,0,26,47,True,26.5; 12.4; 31.7; 45.9; 8.9; 8.9; 67.1; 67.2; ...,True,True,True,True,True,구분 / 전형명 / 변경사항 / 2025 학년도 / 2024 학년도 | 전체 전형 ...
785,U0000046,가톨릭대학교,3,수능 영역별 반영비율/지정응시영역 확인,csat_reflection_ratio,27,high,36,9,9,False,,True,True,True,True,True,전형 / 모집단위 / 수능영역별 반영비율 (%) / 가산점 및 감점 | 국어 / 수...
765,U0000046,가톨릭대학교,4,수능 영역별 반영비율/지정응시영역 확인,csat_reflection_ratio,21,high,2,8,2,True,70; 30; 70; 30; 70; 30; 70; 30,True,True,True,False,False,전형별 주요사항 | 잠재능력우수자 전형 / [ 수능최저학력기준 ] 은 없음 일괄합산...
778,U0000046,가톨릭대학교,5,수능 영역별 반영비율/지정응시영역 확인,csat_reflection_ratio,14,high,20,6,5,False,,True,True,True,False,True,전형명 / 모집단위 / 반영교과 / 교과별반영방법 | 공통 / 일반선택과목 / 진로...


In [6]:
if len(type_summary):
    fig, ax = plt.subplots(figsize=(8.8, 4.8), constrained_layout=True)
    plot_df = type_summary.sort_values("table_candidate_n")
    ax.barh(plot_df["candidate_type"], plot_df["table_candidate_n"], color="#4c78a8")
    ax.set_title(f"Gate 2 rule table candidates by type (n={int(type_summary['table_candidate_n'].sum())})")
    ax.set_xlabel("table candidate count")
    ax.set_ylabel("candidate type")
    save_fig(fig, "fig_gate2_01_candidate_counts_by_type.png")

fig, ax = plt.subplots(figsize=(9, 7.5), constrained_layout=True)
plot_df = priority.head(25).sort_values("manual_priority_score")
ax.barh(plot_df["univ_name_std"], plot_df["manual_priority_score"], color="#e45756")
ax.set_title("Gate 2 manual/external collection priority top 25")
ax.set_xlabel("priority score")
ax.set_ylabel("university")
save_fig(fig, "fig_gate2_02_university_priority.png")

PosixPath('/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_2/data/crawl_2024_admission/gate2_outputs/figures/fig_gate2_02_university_priority.png')

In [7]:
final_summary = pd.DataFrame(
    [
        {"metric": "registry_university_n", "value": registry["univ_id"].nunique()},
        {"metric": "html_cache_n", "value": int(registry["raw_html_exists"].sum())},
        {"metric": "guide_link_candidate_n", "value": len(guide_links)},
        {"metric": "rule_table_candidate_n", "value": len(rule_table_candidates)},
        {"metric": "high_priority_rule_table_n", "value": int(rule_table_candidates["review_priority"].eq("high").sum()) if len(rule_table_candidates) else 0},
        {"metric": "rule_text_snippet_n", "value": len(rule_text_snippets)},
        {"metric": "cache_rule_candidate_university_n", "value": int(priority["rule_cache_status"].eq("CACHE_RULE_CANDIDATES").sum())},
        {"metric": "partial_cache_candidate_university_n", "value": int(priority["rule_cache_status"].eq("PARTIAL_CACHE_CANDIDATES").sum())},
        {"metric": "need_external_guide_university_n", "value": int(priority["rule_cache_status"].eq("NEED_EXTERNAL_GUIDE").sum())},
        {"metric": "external_collection_first_university_n", "value": int(priority["gate2_action"].eq("입학처/모집요강 외부 수집 우선").sum())},
        {"metric": "top_review_queue_row_n", "value": len(review_queue)},
    ]
)
save_csv(final_summary, OUTPUT_DIR / "gate2_06_final_summary.csv")

report = f'''# Gate 2-A/B 모집요강 반영규칙 후보 수집 요약

## 실행 범위

기존 adiga HTML 캐시 51개만 사용했다. 웹 재수집, PDF 다운로드, 최종 반영규칙 확정은 수행하지 않았다.

## 핵심 결과

- HTML 캐시 대학 수: {int(registry["raw_html_exists"].sum()):,}
- 모집요강/입학처 링크 후보: {len(guide_links):,}
- 반영규칙 후보 표: {len(rule_table_candidates):,}
- high priority 후보 표: {int(rule_table_candidates["review_priority"].eq("high").sum()) if len(rule_table_candidates) else 0:,}
- 키워드 주변 snippet: {len(rule_text_snippets):,}
- 캐시에서 수능/산출 규칙 후보를 둘 다 찾은 대학: {int(priority["rule_cache_status"].eq("CACHE_RULE_CANDIDATES").sum()):,}
- 일부 후보만 있는 대학: {int(priority["rule_cache_status"].eq("PARTIAL_CACHE_CANDIDATES").sum()):,}
- 외부 모집요강 우선 수집 필요 대학: {int(priority["rule_cache_status"].eq("NEED_EXTERNAL_GUIDE").sum()):,}

## 생성 파일

- `gate2_00_admission_guide_links.csv`
- `gate2_01_rule_table_candidates.csv`
- `gate2_02_rule_text_snippets.csv`
- `gate2_03_university_rule_collection_priority.csv`
- `gate2_04_candidate_type_summary.csv`
- `gate2_05_gate2_ab_audit_log.csv`
- `gate2_06_final_summary.csv`
- `gate2_07_top_rule_candidates_for_review.csv`

## 다음 판단

`gate2_03_university_rule_collection_priority.csv`의 `gate2_action` 기준으로 외부 모집요강 수집 대상과 캐시 후보 수동검토 대상을 나누면 된다.
'''
report = "\n".join(line.strip() for line in report.splitlines())
report_path = OUTPUT_DIR / "gate2_rule_extraction_summary.md"
report_path.write_text(report, encoding="utf-8")
audit("export", "gate2_rule_extraction_summary.md", "write_markdown", status="PASS", note=str(report_path))
display(final_summary)
display(Markdown(report))

,metric,value
0,registry_university_n,51
1,html_cache_n,51
2,guide_link_candidate_n,102
3,rule_table_candidate_n,1306
4,high_priority_rule_table_n,663
5,rule_text_snippet_n,1008
6,cache_rule_candidate_university_n,51
7,partial_cache_candidate_university_n,0
8,need_external_guide_university_n,0
9,external_collection_first_university_n,0


# Gate 2-A/B 모집요강 반영규칙 후보 수집 요약

## 실행 범위

기존 adiga HTML 캐시 51개만 사용했다. 웹 재수집, PDF 다운로드, 최종 반영규칙 확정은 수행하지 않았다.

## 핵심 결과

- HTML 캐시 대학 수: 51
- 모집요강/입학처 링크 후보: 102
- 반영규칙 후보 표: 1,306
- high priority 후보 표: 663
- 키워드 주변 snippet: 1,008
- 캐시에서 수능/산출 규칙 후보를 둘 다 찾은 대학: 51
- 일부 후보만 있는 대학: 0
- 외부 모집요강 우선 수집 필요 대학: 0

## 생성 파일

- `gate2_00_admission_guide_links.csv`
- `gate2_01_rule_table_candidates.csv`
- `gate2_02_rule_text_snippets.csv`
- `gate2_03_university_rule_collection_priority.csv`
- `gate2_04_candidate_type_summary.csv`
- `gate2_05_gate2_ab_audit_log.csv`
- `gate2_06_final_summary.csv`
- `gate2_07_top_rule_candidates_for_review.csv`

## 다음 판단

`gate2_03_university_rule_collection_priority.csv`의 `gate2_action` 기준으로 외부 모집요강 수집 대상과 캐시 후보 수동검토 대상을 나누면 된다.

In [8]:
audit_log = pd.DataFrame(AUDIT_LOG)
save_csv(audit_log, OUTPUT_DIR / "gate2_05_gate2_ab_audit_log.csv")

required = [
    "gate2_00_admission_guide_links.csv",
    "gate2_01_rule_table_candidates.csv",
    "gate2_02_rule_text_snippets.csv",
    "gate2_03_university_rule_collection_priority.csv",
    "gate2_04_candidate_type_summary.csv",
    "gate2_05_gate2_ab_audit_log.csv",
    "gate2_06_final_summary.csv",
    "gate2_07_top_rule_candidates_for_review.csv",
    "gate2_rule_extraction_summary.md",
    "figures/fig_gate2_01_candidate_counts_by_type.png",
    "figures/fig_gate2_02_university_priority.png",
]
manifest = pd.DataFrame(
    [
        {
            "artifact": name,
            "path": str(OUTPUT_DIR / name),
            "exists": (OUTPUT_DIR / name).exists(),
            "size_bytes": (OUTPUT_DIR / name).stat().st_size if (OUTPUT_DIR / name).exists() else 0,
        }
        for name in required
    ]
)
display(manifest)

,artifact,path,exists,size_bytes
0,gate2_00_admission_guide_links.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,35143
1,gate2_01_rule_table_candidates.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,2113616
2,gate2_02_rule_text_snippets.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,1332525
3,gate2_03_university_rule_collection_priority.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,17551
4,gate2_04_candidate_type_summary.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,359
5,gate2_05_gate2_ab_audit_log.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,2495
6,gate2_06_final_summary.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,347
7,gate2_07_top_rule_candidates_for_review.csv,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,465398
8,gate2_rule_extraction_summary.md,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,1099
9,figures/fig_gate2_01_candidate_counts_by_type.png,/home/sieg/projects-wsl/SBS_dataScience/workbo...,True,45036
